# 使用ddpm生成手写数字

# 去噪模型定义，使用UNet

```
net = Unet()
eps = net(x, t) # 输入噪声图像x和时间步t，给出预测的噪声eps
```

In [ ]:
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


class PositionalEncoding(nn.Module):
    """
    实现了 "Attention is All You Need" 论文中的位置编码层。
    这被用来向模型中注入时间步信息，使其能够知道
    它正在扩散过程的哪个时间点上操作。
    """

    def __init__(self, max_len, num_hiddens=10):
        """
        初始化位置编码模块。
        参数:
            max_len (int): 最大时间步数 (T)。
            num_hiddens (int): 位置编码向量的维度。
        """
        super(PositionalEncoding, self).__init__()
        # 创建一个位置编码矩阵 'P'，形状为 (max_len, num_hiddens)
        P = torch.zeros((max_len, num_hiddens))
        
        # 创建一个位置（时间步）张量 [0, 1, ..., max_len-1]
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / \
            torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
            
        # 对数组中的偶数索引应用 sin 函数; 2i
        P[:, 0::2] = torch.sin(X)
        # 对数组中的奇数索引应用 cos 函数; 2i+1
        P[:, 1::2] = torch.cos(X)
        
        # 将 'P' 注册为缓冲区 (buffer)。缓冲区是模型状态的一部分，
        # 但不被视为需要训练的模型参数。
        self.register_buffer('P', P)

    def forward(self, t):
        """
        为一批时间步检索位置编码。
        参数:
            t (torch.Tensor): 一个形状为 [batch_size] 的张量，包含时间步。
        返回:
            torch.Tensor: 对应的位置编码，形状为 [batch_size, num_hiddens]。
        """
        # 对于输入张量 't' 中的每个时间步，从 'P' 中选择对应的编码向量。
        return self.P[t, :]


class UnetBlock(nn.Module):
    """
    U-Net 架构的一个基本构建块。
    它由两个卷积层和一个残差连接组成。
    在卷积之前应用层归一化 (Layer Normalization)。
    """

    def __init__(self, shape, in_c, out_c):
        """
        初始化 UnetBlock。
        参数:
            shape (tuple): 用于 LayerNorm 的输入张量形状 (channels, height, width)。
            in_c (int): 输入通道数。
            out_c (int): 输出通道数。
        """
        super().__init__()
        self.ln = nn.LayerNorm(shape)
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, stride=1)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, stride=1)
        self.activation = nn.ReLU()
        # 一个 1x1 的卷积层，用于确保残差连接的通道维度
        # 在相加之前与输出通道维度相匹配。
        self.residual_conv = nn.Conv2d(in_c, out_c, kernel_size=1)

    def forward(self, x):
        """
        UnetBlock 的前向传播。
        参数:
            x (torch.Tensor): 输入张量。
        返回:
            torch.Tensor: 输出张量。
        """
        # 应用层归一化
        out = self.ln(x)
        # 第一个卷积和激活函数
        out = self.conv1(out)
        out = self.activation(out)
        # 第二个卷积
        out = self.conv2(out)
        # 添加残差连接 (输入 'x' 通过一个 1x1 卷积)
        out += self.residual_conv(x)
        # 最终的激活函数
        out = self.activation(out)
        return out


class UNet(nn.Module):
    """
    用于预测噪声图像中噪声的 U-Net 模型架构。
    此实现为 DDPM 定制，在网络的每个层级都融入了
    时间步的位置编码。
    """

    def __init__(self,
                 n_steps,
                 channels=[10, 20, 40, 80],
                 pe_dim=10) -> None:
        """
        初始化 UNet 模型。
        参数:
            n_steps (int): 总扩散时间步数 (T)。
            channels (list): U-Net 每个层级的通道数列表。
            pe_dim (int): 时间步位置编码的维度。
        """
        super().__init__()
        # 图像维度 (Channels, Height, Width)，适用于 MNIST
        C, H, W = [1, 28, 28]
        layers = len(channels)
        
        # 计算 U-Net 每个层级的特征图高度和宽度
        Hs = [H]
        Ws = [W]
        cH, cW = H, W
        for _ in range(layers - 1):
            cH //= 2
            cW //= 2
            Hs.append(cH)
            Ws.append(cW)

        # 用于时间步的位置编码
        self.pe = PositionalEncoding(n_steps, pe_dim)

        # --- 编码器 (下采样路径) ---
        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pe_linears_en = nn.ModuleList() # 用于处理每个编码器层级 PE 的线性层
        prev_channel = C
        for i, (channel, cH, cW) in enumerate(zip(channels[0:-1], Hs[0:-1], Ws[0:-1])):
            # 将位置编码投影到正确维度的线性层
            self.pe_linears_en.append(
                nn.Sequential(nn.Linear(pe_dim, prev_channel), nn.ReLU(),
                              nn.Linear(prev_channel, prev_channel)))
            # 编码器块
            self.encoders.append(
                nn.Sequential(
                    UnetBlock((prev_channel, cH, cW), prev_channel, channel),
                    UnetBlock((channel, cH, cW), channel, channel)))
            # 下采样层
            self.downs.append(nn.Conv2d(channel, channel, kernel_size=2, stride=2))
            prev_channel = channel

        # --- 瓶颈层 (中间部分) ---
        self.pe_mid = nn.Linear(pe_dim, prev_channel)
        channel = channels[-1]
        self.mid = nn.Sequential(
            UnetBlock((prev_channel, Hs[-1], Ws[-1]), prev_channel, channel),
            UnetBlock((channel, Hs[-1], Ws[-1]), channel, channel),
        )

        # --- 解码器 (上采样路径) ---
        self.decoders = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pe_linears_de = nn.ModuleList() # 用于处理每个解码器层级 PE 的线性层
        prev_channel = channel
        for i, (channel, cH, cW) in enumerate(zip(channels[-2::-1], Hs[-2::-1], Ws[-2::-1])):
            # 用于位置编码的线性层
            self.pe_linears_de.append(nn.Linear(pe_dim, prev_channel))
            # 上采样层
            self.ups.append(nn.ConvTranspose2d(prev_channel, channel, kernel_size=2, stride=2))
            # 解码器块。由于与跳跃连接进行拼接，输入通道数加倍。
            self.decoders.append(
                nn.Sequential(
                    UnetBlock((channel * 2, cH, cW), channel * 2, channel),
                    UnetBlock((channel, cH, cW), channel, channel)))
            prev_channel = channel

        # 最后的输出卷积层，将特征图映射回原始图像通道数
        self.conv_out = nn.Conv2d(prev_channel, C, kernel_size=3, padding=1, stride=1)

    def forward(self, x, t):
        """
        UNet 的前向传播。
        参数:
            x (torch.Tensor): 带噪的输入图像张量，形状为 [batch, C, H, W]。
            t (torch.Tensor): 时间步张量，形状为 [batch]。
        返回:
            torch.Tensor: 预测的噪声，与 x 形状相同。
        """
        n = t.shape[0]
        # 获取时间步的位置编码
        t = self.pe(t)
        
        encoder_outs = []
        # --- 编码器路径 ---
        for pe_linear, encoder, down in zip(self.pe_linears_en, self.encoders, self.downs):
            # 处理并添加位置编码
            pe = pe_linear(t).reshape(n, -1, 1, 1)
            x = encoder(x + pe)
            # 存储输出以用于跳跃连接
            encoder_outs.append(x)
            # 下采样
            x = down(x)
            
        # --- 瓶颈层 ---
        pe = self.pe_mid(t).reshape(n, -1, 1, 1)
        x = self.mid(x + pe)
        
        # --- 解码器路径 ---
        for pe_linear, decoder, up, encoder_out in zip(self.pe_linears_de, self.decoders, self.ups, encoder_outs[::-1]):
            # 处理并添加位置编码
            pe = pe_linear(t).reshape(n, -1, 1, 1)
            # 上采样
            x = up(x)

            # 如果上采样后的特征图尺寸小于跳跃连接的尺寸，则进行填充
            pad_x = encoder_out.shape[2] - x.shape[2]
            pad_y = encoder_out.shape[3] - x.shape[3]
            x = F.pad(x, (pad_x // 2, pad_x - pad_x // 2, pad_y // 2, pad_y - pad_y // 2))
            
            # 拼接跳跃连接
            x = torch.cat((encoder_out, x), dim=1)
            # 应用解码器块
            x = decoder(x + pe) # 在这里添加 PE 是一种设计选择
            
        # 最终输出层
        x = self.conv_out(x)
        return x

# DDPM调度逻辑

In [ ]:

class DDPM():
    """
    实现了去噪扩散概率模型 (DDPM) 的逻辑。
    这个类管理扩散调度表 (betas, alphas)，并提供
    前向 (加噪) 和反向 (去噪/采样) 过程的方法。
    """
    def __init__(self,
                 device,
                 n_steps: int,
                 min_beta: float = 0.0001,
                 max_beta: float = 0.02):
        """
        初始化 DDPM 调度器。
        参数:
            device: torch 设备 ('cuda' 或 'cpu')。
            n_steps (int): 总扩散步数 (T)。
            min_beta (float): beta 的起始值。
            max_beta (float): beta 的结束值。
        """
        # 定义 beta 调度表 (每一步添加的噪声方差)，呈线性增长
        betas = torch.linspace(min_beta, max_beta, n_steps).to(device)
        # 基于 betas 计算 alphas
        alphas = 1 - betas
        # 计算 alpha_bars (alphas 的累积乘积)
        alpha_bars = torch.cumprod(alphas, dim=0)

        self.betas = betas
        self.n_steps = n_steps
        self.alphas = alphas
        self.alpha_bars = alpha_bars

    def sample_forward(self, x, t, eps=None):
        """
        执行前向扩散过程 (加噪)。
        它根据给定的时间步 't' 向图像 'x' 中添加噪声。
        这基于公式: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
        参数:
            x (torch.Tensor): 初始的干净图像 (x_0)。
            t (torch.Tensor): 时间步。
            eps (torch.Tensor, optional): 要添加的噪声。如果为 None，则会自动生成。
        返回:
            torch.Tensor: 加噪后的图像 (x_t)。
        """
        # 获取给定时间步的 alpha_bar 值
        alpha_bar = self.alpha_bars[t].reshape(-1, 1, 1, 1)
        if eps is None:
            eps = torch.randn_like(x)
        # 应用前向过程公式
        res = torch.sqrt(alpha_bar) * x + torch.sqrt(1 - alpha_bar) * eps
        return res

    def sample_backward(self, img_shape, net, device, simple_var=True):
        """
        执行完整的反向过程 (采样/去噪)。
        从纯噪声开始，通过 T 步迭代去噪，生成一张图像。
        参数:
            img_shape (tuple): 要生成的图像形状 (例如, [batch, C, H, W])。
            net (nn.Module): 用于噪声预测的 U-Net 模型。
            device: torch 设备。
            simple_var (bool): 如果为 True，使用 beta_t 作为方差。如果为 False，使用一个更复杂的公式。
        返回:
            torch.Tensor: 生成的图像。
        """
        # 从纯高斯噪声开始 (x_T)
        x = torch.randn(img_shape).to(device)
        # 从 T-1 到 0 反向循环
        for t in range(self.n_steps - 1, -1, -1):
            x = self.sample_backward_step(x, t, net, simple_var)
        return x

    def sample_backward_step(self, x_t, t, net, simple_var=True):
        """
        执行单步反向去噪过程。
        它预测 x_t 中的噪声，并用它来估计 x_{t-1}。
        参数:
            x_t (torch.Tensor): 在步骤 t 的带噪图像。
            t (int): 当前时间步。
            net (nn.Module): U-Net 模型。
            simple_var (bool): 控制方差的计算方式。
        返回:
            torch.Tensor: 噪声更少的图像 x_{t-1}。
        """
        n = x_t.shape[0]
        # 为整个批次创建当前时间步的张量
        t_tensor = torch.tensor([t] * n, dtype=torch.long).to(x_t.device)
        
        # 使用 U-Net 预测噪声 (epsilon)
        eps = net(x_t, t_tensor)
        
        if t == 0:
            noise = 0
        else:
            # 计算噪声项的方差
            if simple_var:
                var = self.betas[t]
            else:
                # 论文中一个更复杂的方差计算方法
                var = (1 - self.alpha_bars[t - 1]) / (1 - self.alpha_bars[t]) * self.betas[t]
            
            # 为这一步生成随机噪声
            noise = torch.randn_like(x_t)
            noise *= torch.sqrt(var)

        # 计算 x_{t-1} 分布的均值
        # 这源自 DDPM 论文中的去噪公式
        mean = (x_t - (1 - self.alphas[t]) / torch.sqrt(1 - self.alpha_bars[t]) * eps) / torch.sqrt(self.alphas[t])
        
        # 新的、噪声更少的图像是均值加上噪声项
        x_t_minus_1 = mean + noise

        return x_t_minus_1


# 自定义的数据集加载器

In [ ]:

class DigitRecognizerDataset(Dataset):
    """用于从 CSV 文件加载 MNIST 数据的自定义 PyTorch 数据集。"""
    def __init__(self, csv_file, labels=True, transform=None):
        df = pd.read_csv(csv_file)
        if labels:
            self.labels = df.iloc[:, 0].values
            # 分离像素数据，重塑为 [N, C, H, W] 格式，并归一化到 [0, 1] 范围
            self.data = df.iloc[:, 1:].values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
        else:
            self.labels = None
            self.data = df.values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = torch.tensor(self.data[idx])
        if self.transform:
             image = self.transform(image)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return image, label
        return image


# 训练

In [ ]:
# 设置随机种子以确保可复现性
np.random.seed(47)
torch.manual_seed(47)

# 一个标准的 MNIST 转换。
# Normalize 转换会将范围从 [0,1] 移到 [-1,1]。
transform = transforms.Compose([
    transforms.Normalize((0.5,), (0.5,))
])

data_path = '/kaggle/input/digit-recognizer/train.csv'
mnist = DigitRecognizerDataset(data_path, True, transform=None)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)

# --- 训练超参数 ---
batch_size = 64
lr = 1e-3
num_epochs = 10
n_steps = 1000 # 总扩散步数

data_loader = DataLoader(mnist, batch_size=batch_size, shuffle=True)

# --- 模型初始化 ---
net = UNet(n_steps).to(device)
ddpm = DDPM(device, n_steps)
optimizer = optim.Adam(net.parameters(), lr=lr)
# 损失函数是真实噪声和预测噪声之间的均方误差 (MSE)。
loss_fn = nn.MSELoss()

# --- 训练循环 ---
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for x, _ in data_loader:
        current_batch_size = x.shape[0]
        # 数据加载时的范围是 [0, 1]，我们将其调整到 [-1, 1]，这在扩散模型中很常见
        x = (x * 2) - 1
        x = x.to(device)
        
        # 1. 为批次中的每个图像采样一个随机时间步 't'
        t = torch.randint(0, n_steps, (current_batch_size,)).to(device)
        # 2. 生成与图像形状相同的随机噪声 'eps'
        eps = torch.randn_like(x).to(device)
        # 3. 使用前向过程创建带噪图像 'x_t'
        x_t = ddpm.sample_forward(x, t, eps)
        # 4. 使用 U-Net 预测 'x_t' 中的噪声
        eps_theta = net(x_t, t)
        # 5. 计算真实噪声和预测噪声之间的损失
        loss = loss_fn(eps_theta, eps)
        
        # 标准的反向传播步骤
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * current_batch_size
        
    print(f'Epoch {epoch + 1} loss: {epoch_loss / len(mnist):.4f}')


# **生成与可视化**

In [ ]:

with torch.no_grad():
    # --- 生成最终样本的网格图 ---
    num_samples = 9
    shape = (num_samples, 1, 28, 28)

    # 通过运行完整的反向采样过程来生成图像
    images = ddpm.sample_backward(shape, net, device=device)
    # 为了可视化，将图像范围从 [-1, 1] 缩放回 [0, 1]
    images = (images + 1) / 2
    images = images.clamp(0, 1).detach().cpu().numpy()

    fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < num_samples:
            ax.imshow(images[i][0], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()


# **可视化整个去噪过程**

In [ ]:
with torch.no_grad():
    # --- 可视化整个去噪过程 ---
    shape = (1, 1, 28, 28)
    x = torch.randn(shape).to(device)
    out = []
    step_interval = 50 # 每 50 步存储一次图像

    for t in range(n_steps - 1, -1, -1):
        x = ddpm.sample_backward_step(x, t, net)
        if t % step_interval == 0 or t == 0:
            # 为了绘图，将范围从 [-1,1] 缩放至 [0,1]
            x_norm = (x + 1) / 2
            x_norm = x_norm.clamp(0, 1).detach().cpu().numpy()
            out.append((t, x_norm[0, 0]))

    n_steps_plot = len(out)
    n_cols = 4
    n_rows = (n_steps_plot + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for i, (ax, (t, img)) in enumerate(zip(axes.flat, out)):
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Step {t}', fontsize=10)
        ax.axis('off')

    for j in range(i + 1, n_rows * n_cols):
        axes.flat[j].axis('off')

    plt.tight_layout()
    plt.suptitle('Denoising Process (Every 50 Steps)', y=1.02)
    plt.show()